In [8]:
from IPython.display import display

import sys
import time

import numba
import numpy as np
import pandas as pd

sys.path.insert(0, "src")

In [9]:
@numba.njit
def _lorenz(x_var, t, params):
    x, y, z = x_var
    a, b, c = params
    dx_dt = a * (y - x)
    dy_dt = x * (b - z) - y
    dz_dt = x * y - c * z

    return np.array([dx_dt, dy_dt, dz_dt])

ic = np.array([0.0, 1.5, 15.0], dtype=np.float64)
p = np.ascontiguousarray([10.0, 28.0, 8 / 3], dtype=np.float64)
t_min, t_max = 0, 50
n_default = 1000000

## Pure Python

In [10]:
def _numerical_jacobian_py(x, y, z, eq, params, eps=1e-6):
    xyz = np.array([x, y, z])
    f0 = eq(xyz, 0.0, params)
    J = np.zeros((3, 3))
    J[:, 0] = (eq(np.array([x + eps, y, z]), 0.0, params) - f0) / eps
    J[:, 1] = (eq(np.array([x, y + eps, z]), 0.0, params) - f0) / eps
    J[:, 2] = (eq(np.array([x, y, z + eps]), 0.0, params) - f0) / eps

    return J


def _augmented_rhs_py(state, params, eq):
    x, y, z = state[0], state[1], state[2]
    dxdydz = eq(np.array([x, y, z]), 0.0, params)
    J = _numerical_jacobian_py(x, y, z, eq, params)
    theta = state[3:].reshape(3, 3)
    out = np.zeros(12)
    out[0], out[1], out[2] = dxdydz[0], dxdydz[1], dxdydz[2]
    out[3:] = (J @ theta).ravel()

    return out


def _rk4_step_py(state, dt, params, eq):
    k1 = _augmented_rhs_py(state, params, eq)
    k2 = _augmented_rhs_py(state + 0.5 * dt * k1, params, eq)
    k3 = _augmented_rhs_py(state + 0.5 * dt * k2, params, eq)
    k4 = _augmented_rhs_py(state + dt * k3, params, eq)

    return state + (dt / 6.0) * (k1 + 2 * k2 + 2 * k3 + k4)


def _gram_schmidt_py(theta_flat):
    lyap_sums = np.zeros(3)
    theta = theta_flat.reshape(3, 3).copy()

    for i in range(3):
        for j in range(i):
            dot = theta[0, i] * theta[0, j] + theta[1, i] * theta[1, j] + theta[2, i] * theta[2, j]
            for k in range(3):
                theta[k, i] -= dot * theta[k, j]

        norm = np.sqrt(theta[0, i] ** 2 + theta[1, i] ** 2 + theta[2, i] ** 2)
        lyap_sums[i] = np.log(norm)

        for k in range(3):
            theta[k, i] /= norm

    return theta.ravel(), lyap_sums


def compute_lyapunov_py(
    equation, initial_conditions, params, t_min, t_max, n, gs_interval=10,
):
    state = np.zeros(12, dtype=np.float64)
    state[:3] = initial_conditions
    state[3:] = np.eye(3).ravel()
    dt = (t_max - t_min) / n
    lyap_sums = np.zeros(3)
    n_gs = n // gs_interval

    t_hist = np.empty(n_gs)
    lyap_hist = np.empty((n_gs, 3))

    gs_counter = 0

    for i in range(n):
        state = _rk4_step_py(state, dt, params, equation)

        if (i + 1) % gs_interval == 0:
            state[3:], sums = _gram_schmidt_py(state[3:])
            lyap_sums += sums
            gs_counter += 1

            t_current = gs_counter * gs_interval * dt
            t_hist[gs_counter - 1] = t_current
            lyap_hist[gs_counter - 1] = lyap_sums / t_current

    total_gs_time = gs_counter * gs_interval * dt
    lyap = lyap_sums / total_gs_time

    if lyap[0] + lyap[1] >= 0 and lyap[2] < 0:
        ky = 2.0 + (lyap[0] + lyap[1]) / abs(lyap[2])
    elif lyap[0] < 0:
        ky = 0.0
    else:
        ky = 3.0

    return lyap, ky, t_hist[:gs_counter], lyap_hist[:gs_counter]

## JIT compiled

In [11]:
@numba.njit(nogil=True)
def _numerical_jacobian(x, y, z, eq, params, eps=1e-6):
    xyz = np.array([x, y, z])
    f0 = eq(xyz, 0.0, params)
    J = np.zeros((3, 3))
    J[:, 0] = (eq(np.array([x + eps, y, z]), 0.0, params) - f0) / eps
    J[:, 1] = (eq(np.array([x, y + eps, z]), 0.0, params) - f0) / eps
    J[:, 2] = (eq(np.array([x, y, z + eps]), 0.0, params) - f0) / eps

    return J


@numba.njit(nogil=True)
def _augmented_rhs(state, params, eq):
    x, y, z = state[0], state[1], state[2]
    dxdydz = eq(np.array([x, y, z]), 0.0, params)
    J = _numerical_jacobian(x, y, z, eq, params)
    theta = state[3:].reshape(3, 3)
    out = np.zeros(12)
    out[0], out[1], out[2] = dxdydz[0], dxdydz[1], dxdydz[2]
    out[3:] = (J @ theta).ravel()

    return out


@numba.njit(nogil=True)
def _rk4_step(state, dt, params, eq):
    k1 = _augmented_rhs(state, params, eq)
    k2 = _augmented_rhs(state + 0.5 * dt * k1, params, eq)
    k3 = _augmented_rhs(state + 0.5 * dt * k2, params, eq)
    k4 = _augmented_rhs(state + dt * k3, params, eq)

    return state + (dt / 6.0) * (k1 + 2 * k2 + 2 * k3 + k4)


@numba.njit(nogil=True)
def _gram_schmidt(theta_flat):
    lyap_sums = np.zeros(3)
    theta = theta_flat.reshape(3, 3).copy()

    for i in range(3):
        for j in range(i):
            dot = (
                theta[0, i] * theta[0, j]
                + theta[1, i] * theta[1, j]
                + theta[2, i] * theta[2, j]
            )
            for k in range(3):
                theta[k, i] -= dot * theta[k, j]

        norm = np.sqrt(theta[0, i] ** 2 + theta[1, i] ** 2 + theta[2, i] ** 2)
        lyap_sums[i] = np.log(norm)

        for k in range(3):
            theta[k, i] /= norm

    return theta.ravel(), lyap_sums


@numba.njit(nogil=True)
def compute_lyapunov_jit(
    equation, initial_conditions, params, t_min, t_max, n, gs_interval=10,
):
    state = np.zeros(12, dtype=np.float64)
    state[:3] = initial_conditions
    state[3:] = np.eye(3).ravel()
    dt = (t_max - t_min) / n
    lyap_sums = np.zeros(3)

    n_gs = n // gs_interval
    t_hist = np.empty(n_gs)
    lyap_hist = np.empty((n_gs, 3))

    gs_counter = 0

    for i in range(n):
        state = _rk4_step(state, dt, params, equation)

        if (i + 1) % gs_interval == 0:
            state[3:], sums = _gram_schmidt(state[3:])
            lyap_sums += sums
            gs_counter += 1

            t_current = gs_counter * gs_interval * dt
            t_hist[gs_counter - 1] = t_current
            lyap_hist[gs_counter - 1] = lyap_sums / t_current

    total_gs_time = gs_counter * gs_interval * dt
    lyap = lyap_sums / total_gs_time

    if lyap[0] + lyap[1] >= 0 and lyap[2] < 0:
        ky = 2.0 + (lyap[0] + lyap[1]) / abs(lyap[2])
    elif lyap[0] < 0:
        ky = 0.0
    else:
        ky = 3.0

    return lyap, ky, t_hist[:gs_counter], lyap_hist[:gs_counter]

In [12]:
print("Warming up JIT")
lyap_j, ky_j, _, _ = compute_lyapunov_jit(_lorenz, ic, p, t_min, t_max, 100000)
print(f"JIT warmed up: lyap={lyap_j}  ky={ky_j:.4f}")

print("Verifying at low resolution")
lyap_p, ky_p, _, _ = compute_lyapunov_py(_lorenz, ic, p, t_min, t_max, 5000)
print(f"Py check done: lyap={lyap_p}  ky={ky_p:.4f}")

Warming up JIT
JIT warmed up: lyap=[  0.88989268  -0.05186616 -14.50469319]  ky=2.0578
Verifying at low resolution
Py check done: lyap=[  0.94709472  -0.062581   -14.5510769 ]  ky=2.0608


## JIT and pure Python comparison

Pure Python at n=1M would take ages so benchmark at smaller n and extrapolate

In [13]:
n_warmup = 2
n_bench = 3
ns = [5000, 10000, 20000, 50000]

dfs = {}

for label, fn in [
    ("Py", compute_lyapunov_py),
    ("JIT", compute_lyapunov_jit),
]:
    print(f"Running {label} benchmarks")

    records = []
    for n in ns:
        for _ in range(n_warmup):
            fn(_lorenz, ic, p, t_min, t_max, n)

        times = []
        for _ in range(n_bench):
            t0 = time.perf_counter()
            lyap, ky, _, _ = fn(_lorenz, ic, p, t_min, t_max, n)
            times.append(time.perf_counter() - t0)

        t_mean = np.mean(times)

        records.append({
            "n": n,
            "time (s)": t_mean,
            "lyap[0]": lyap[0],
            "lyap[1]": lyap[1],
            "lyap[2]": lyap[2],
            "ky": ky
        })

    dfs[label] = pd.DataFrame(records)

print("\nPython")
display(dfs["Py"])
print("\nJIT")
display(dfs["JIT"])

Running Py benchmarks
Running JIT benchmarks

Python


,n,time (s),lyap[0],lyap[1],lyap[2],ky
0,5000,0.220899,0.947095,-0.062581,-14.551077,2.060787
1,10000,0.445555,0.923188,-0.051456,-14.538392,2.059961
2,20000,0.910884,0.935360,-0.054043,-14.547983,2.060580
3,50000,2.342972,0.877724,-0.041194,-14.503197,2.057679



JIT


,n,time (s),lyap[0],lyap[1],lyap[2],ky
0,5000,0.014475,0.947095,-0.062581,-14.551077,2.060787
1,10000,0.028895,0.923188,-0.051456,-14.538392,2.059961
2,20000,0.057894,0.935360,-0.054043,-14.547983,2.060580
3,50000,0.144669,0.877724,-0.041194,-14.503197,2.057679


## Summary

In [14]:
n = 50000
for _ in range(3):
    compute_lyapunov_py(_lorenz, ic, p, t_min, t_max, n)
    compute_lyapunov_jit(_lorenz, ic, p, t_min, t_max, n)

t0 = time.perf_counter()
lyap_p, ky_p, _, _ = compute_lyapunov_py(_lorenz, ic, p, t_min, t_max, n)
t_py = time.perf_counter() - t0

t0 = time.perf_counter()
lyap_j, ky_j, _, _ = compute_lyapunov_jit(_lorenz, ic, p, t_min, t_max, n)
t_jit = time.perf_counter() - t0

print(f"n={n}")
print(f"Python:      {t_py:.4f}s")
print(f"JIT:         {t_jit:.4f}s")
print(f"Speedup:     {t_py / t_jit:.1f}x")
print("\nExtrapolated to n=1000000:")
print(f"Python:      approx {t_py * 20:.1f}s")
print(f"JIT:         approx {t_jit * 20:.2f}s")

n=50000
Python:      2.4770s
JIT:         0.1467s
Speedup:     16.9x

Extrapolated to n=1000000:
Python:      approx 49.5s
JIT:         approx 2.93s
